# South Carolina H.4216 Tax Reform Analysis (Tax Year 2026)

This notebook analyzes the impact of SC H.4216 tax reform.

## Proposal
- Apply a tax rate of 1.99% on taxable income up to $30,000 and 5.39% over
- Eliminate the federal standard or itemized deduction
- Allow a new SC Income Adjusted Deduction (SCIAD) at certain income levels
- Maintain all other state adjustments, exemptions, and credits
- Cap SC EITC at $200

## Current 2026 Marginal Tax Rates
- 0% up to $3,640
- 3% $3,640 - $18,230
- 6% over $18,230

## Proposed Tax Rates
- 1.99% up to $30,000
- 5.39% over $30,000

## SC Deduction (SCIAD) Phase-out
| Filing Status | Amount | Phase-out Start | Phase-out End |
|---------------|--------|-----------------|---------------|
| Single | $15,000 | $40,000 | $95,000 |
| Married Joint | $30,000 | $80,000 | $190,000 |
| Head of Household | $22,500 | $60,000 | $142,500 |

## Implementation Note
This analysis uses the corrected H.4216 implementation (PR #7514) which properly handles SC additions.
The fix removes `sc_additions` from the H.4216 taxable income formula since H.4216 starts from AGI
(before federal deductions), making addbacks for QBI and SALT inappropriate.

In [ ]:
from policyengine_us import Microsimulation
from policyengine_us.reforms.states.sc.h4216.sc_h4216 import create_sc_h4216
from policyengine_core.reforms import Reform
import pandas as pd
import numpy as np

SC_DATASET = "hf://policyengine/policyengine-us-data/states/SC.h5"
TAX_YEAR = 2026  # Renamed to avoid conflict with YEAR constant from model_api

In [ ]:
from policyengine_us.model_api import *

def create_h4216_reform():
    """
    SC H.4216 Reform:
    - Enable H.4216 via in_effect parameter
    - Set rates: 1.99% up to $30k, 5.39% over $30k
    """
    # Parameter changes via Reform.from_dict
    param_reform = Reform.from_dict(
        {
            "gov.contrib.states.sc.h4216.in_effect": {
                "2026-01-01.2100-12-31": True
            },
            "gov.contrib.states.sc.h4216.rates[1].rate": {
                "2026-01-01.2100-12-31": 0.0539
            }
        },
        country_id="us",
    )
    
    # Get base H.4216 reform (EITC cap, SCIAD, taxable income, tax calculation)
    base_reform = create_sc_h4216()
    
    # Order: base reform first, then parameter overrides
    return (base_reform, param_reform)

print("Reform function defined!")

In [ ]:
print("Loading baseline (current SC tax law)...")
baseline = Microsimulation(dataset=SC_DATASET)
print("Baseline loaded")

print("\nLoading reform (H.4216 with 5.39% top rate)...")
reform = create_h4216_reform()
reform_sim = Microsimulation(dataset=SC_DATASET, reform=reform)
print("Reform loaded")

print("\n" + "="*60)
print("All simulations ready!")
print("="*60)

## Calculate Tax Impacts by Income Bracket

In [ ]:
# Get tax unit level data
baseline_tax = np.array(baseline.calculate("sc_income_tax", period=TAX_YEAR, map_to="tax_unit"))
reform_tax = np.array(reform_sim.calculate("sc_income_tax", period=TAX_YEAR, map_to="tax_unit"))
agi = np.array(baseline.calculate("adjusted_gross_income", period=TAX_YEAR, map_to="tax_unit"))
tax_unit_weight = np.array(baseline.calculate("tax_unit_weight", period=TAX_YEAR))

# Calculate tax change
tax_change = reform_tax - baseline_tax

print(f"Total tax units: {len(baseline_tax):,}")
print(f"Weighted tax units (returns): {tax_unit_weight.sum():,.0f}")

In [ ]:
# Define income brackets matching the RFA analysis
income_brackets = [
    (float('-inf'), 0, "$0*"),
    (0, 10000, "$1 to $10,000"),
    (10000, 20000, "$10,001 to $20,000"),
    (20000, 30000, "$20,001 to $30,000"),
    (30000, 40000, "$30,001 to $40,000"),
    (40000, 50000, "$40,001 to $50,000"),
    (50000, 75000, "$50,001 to $75,000"),
    (75000, 100000, "$75,001 to $100,000"),
    (100000, 150000, "$100,001 to $150,000"),
    (150000, 200000, "$150,001 to $200,000"),
    (200000, 300000, "$200,001 to $300,000"),
    (300000, 500000, "$300,001 to $500,000"),
    (500000, 1000000, "$500,001 to $1,000,000"),
    (1000000, float('inf'), "Over $1,000,000")
]

results = []

for lower, upper, label in income_brackets:
    if lower == float('-inf'):
        mask = agi <= upper
    elif upper == float('inf'):
        mask = agi > lower
    else:
        mask = (agi > lower) & (agi <= upper)
    
    if mask.sum() == 0:
        continue
    
    # Weighted counts
    est_returns = tax_unit_weight[mask].sum()
    pct_returns = est_returns / tax_unit_weight.sum() * 100
    
    # Tax liability
    old_avg_tax = np.average(baseline_tax[mask], weights=tax_unit_weight[mask]) if est_returns > 0 else 0
    new_avg_tax = np.average(reform_tax[mask], weights=tax_unit_weight[mask]) if est_returns > 0 else 0
    
    # Returns with tax change (threshold of $1)
    change_mask = mask & (np.abs(tax_change) > 1)
    returns_with_change = tax_unit_weight[change_mask].sum()
    pct_with_change = returns_with_change / est_returns * 100 if est_returns > 0 else 0
    
    if returns_with_change > 0:
        old_avg_tax_changed = np.average(baseline_tax[change_mask], weights=tax_unit_weight[change_mask])
        new_avg_tax_changed = np.average(reform_tax[change_mask], weights=tax_unit_weight[change_mask])
        avg_change = new_avg_tax_changed - old_avg_tax_changed
    else:
        old_avg_tax_changed = 0
        new_avg_tax_changed = 0
        avg_change = 0
    
    total_change = (tax_change[mask] * tax_unit_weight[mask]).sum()
    
    # Tax decrease
    decrease_mask = mask & (tax_change < -1)
    decrease_returns = tax_unit_weight[decrease_mask].sum()
    decrease_pct = decrease_returns / est_returns * 100 if est_returns > 0 else 0
    total_decrease = (tax_change[decrease_mask] * tax_unit_weight[decrease_mask]).sum() if decrease_returns > 0 else 0
    avg_decrease = np.average(tax_change[decrease_mask], weights=tax_unit_weight[decrease_mask]) if decrease_returns > 0 else 0
    
    # Tax increase
    increase_mask = mask & (tax_change > 1)
    increase_returns = tax_unit_weight[increase_mask].sum()
    increase_pct = increase_returns / est_returns * 100 if est_returns > 0 else 0
    total_increase = (tax_change[increase_mask] * tax_unit_weight[increase_mask]).sum() if increase_returns > 0 else 0
    avg_increase = np.average(tax_change[increase_mask], weights=tax_unit_weight[increase_mask]) if increase_returns > 0 else 0
    
    # No change
    no_change_mask = mask & (np.abs(tax_change) <= 1)
    no_change_returns = tax_unit_weight[no_change_mask].sum()
    no_change_pct = no_change_returns / est_returns * 100 if est_returns > 0 else 0
    
    # Zero tax liability (under reform)
    zero_tax_mask = mask & (reform_tax <= 0)
    zero_tax_returns = tax_unit_weight[zero_tax_mask].sum()
    zero_tax_pct = zero_tax_returns / est_returns * 100 if est_returns > 0 else 0
    
    results.append({
        "Federal AGI Range": label,
        "Est. # Returns": int(round(est_returns)),
        "% of Returns": round(pct_returns, 1),
        "Old Avg Tax": int(round(old_avg_tax)),
        "New Avg Tax": int(round(new_avg_tax)),
        "Returns w/ Change": int(round(returns_with_change)),
        "% w/ Change": round(pct_with_change, 1),
        "Avg Change": int(round(avg_change)),
        "Total Change ($)": int(round(total_change)),
        "Decrease #": int(round(decrease_returns)),
        "Decrease %": round(decrease_pct, 1),
        "Total Decrease ($)": int(round(total_decrease)),
        "Avg Decrease": int(round(avg_decrease)),
        "Increase #": int(round(increase_returns)),
        "Increase %": round(increase_pct, 1),
        "Total Increase ($)": int(round(total_increase)),
        "Avg Increase": int(round(avg_increase)),
        "No Change #": int(round(no_change_returns)),
        "No Change %": round(no_change_pct, 1),
        "Zero Tax #": int(round(zero_tax_returns)),
        "Zero Tax %": round(zero_tax_pct, 1)
    })

df_results = pd.DataFrame(results)
print("Results calculated!")

In [ ]:
# Calculate totals
total_returns = tax_unit_weight.sum()
total_old_tax = np.average(baseline_tax, weights=tax_unit_weight)
total_new_tax = np.average(reform_tax, weights=tax_unit_weight)

change_mask_all = np.abs(tax_change) > 1
total_returns_changed = tax_unit_weight[change_mask_all].sum()
total_change_amount = (tax_change * tax_unit_weight).sum()

decrease_mask_all = tax_change < -1
total_decrease_returns = tax_unit_weight[decrease_mask_all].sum()
total_decrease_amount = (tax_change[decrease_mask_all] * tax_unit_weight[decrease_mask_all]).sum()

increase_mask_all = tax_change > 1
total_increase_returns = tax_unit_weight[increase_mask_all].sum()
total_increase_amount = (tax_change[increase_mask_all] * tax_unit_weight[increase_mask_all]).sum()

no_change_mask_all = np.abs(tax_change) <= 1
total_no_change_returns = tax_unit_weight[no_change_mask_all].sum()

zero_tax_mask_all = reform_tax <= 0
total_zero_tax_returns = tax_unit_weight[zero_tax_mask_all].sum()

# Add totals row
totals = {
    "Federal AGI Range": "Total",
    "Est. # Returns": int(round(total_returns)),
    "% of Returns": 100.0,
    "Old Avg Tax": int(round(total_old_tax)),
    "New Avg Tax": int(round(total_new_tax)),
    "Returns w/ Change": int(round(total_returns_changed)),
    "% w/ Change": round(total_returns_changed / total_returns * 100, 1),
    "Avg Change": int(round(total_new_tax - total_old_tax)),
    "Total Change ($)": int(round(total_change_amount)),
    "Decrease #": int(round(total_decrease_returns)),
    "Decrease %": round(total_decrease_returns / total_returns * 100, 1),
    "Total Decrease ($)": int(round(total_decrease_amount)),
    "Avg Decrease": int(round(total_decrease_amount / total_decrease_returns)) if total_decrease_returns > 0 else 0,
    "Increase #": int(round(total_increase_returns)),
    "Increase %": round(total_increase_returns / total_returns * 100, 1),
    "Total Increase ($)": int(round(total_increase_amount)),
    "Avg Increase": int(round(total_increase_amount / total_increase_returns)) if total_increase_returns > 0 else 0,
    "No Change #": int(round(total_no_change_returns)),
    "No Change %": round(total_no_change_returns / total_returns * 100, 1),
    "Zero Tax #": int(round(total_zero_tax_returns)),
    "Zero Tax %": round(total_zero_tax_returns / total_returns * 100, 1)
}

df_results = pd.concat([df_results, pd.DataFrame([totals])], ignore_index=True)

## Results Summary

In [ ]:
print("="*100)
print("H. 4216 - ESTIMATED SOUTH CAROLINA INDIVIDUAL INCOME TAX IMPACT")
print(f"Tax Year {TAX_YEAR}")
print("="*100)
print(f"\nProposal: Apply a tax rate of 1.99% on taxable income up to $30,000 and 5.39% over,")
print(f"eliminate the federal standard or itemized deduction, allow a new SC deduction at")
print(f"certain income levels, and maintain all other state adjustments, exemptions, and credits.")
print("="*100)

# Summary stats
pct_decrease = total_decrease_returns / total_returns * 100
pct_increase = total_increase_returns / total_returns * 100
pct_unchanged = total_no_change_returns / total_returns * 100

print(f"\nImpact: With this tax structure:")
print(f"  - {pct_decrease:.1f}% of taxpayers have a LOWER tax liability")
print(f"  - {pct_increase:.1f}% of taxpayers have a HIGHER tax liability")
print(f"  - {pct_unchanged:.1f}% are UNCHANGED")
print(f"\nGeneral Fund Impact: ${total_change_amount:,.0f}")
print("="*100)

In [ ]:
# Display main results table
display_cols = [
    "Federal AGI Range", "Est. # Returns", "% of Returns", 
    "Old Avg Tax", "New Avg Tax", "Total Change ($)",
    "Decrease #", "Decrease %", "Increase #", "Increase %",
    "No Change %", "Zero Tax %"
]

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}' if isinstance(x, float) else x)

print(df_results[display_cols].to_string(index=False))

In [ ]:
# Export full results
df_results.to_csv('sc_h4216_tax_impact_analysis.csv', index=False)
print("\nFull results exported to: sc_h4216_tax_impact_analysis.csv")

## Detailed Breakdown Tables

In [ ]:
# Tax Return Distribution
print("\n" + "="*80)
print("ESTIMATED TAX RETURN DISTRIBUTION")
print("="*80)
dist_cols = ["Federal AGI Range", "Est. # Returns", "% of Returns", "Old Avg Tax", "New Avg Tax"]
print(df_results[dist_cols].to_string(index=False))

In [ ]:
# Tax Decrease Summary
print("\n" + "="*80)
print("TAX RETURNS WITH A DECREASE IN LIABILITY")
print("="*80)
decrease_cols = ["Federal AGI Range", "Decrease #", "Decrease %", "Total Decrease ($)", "Avg Decrease"]
print(df_results[decrease_cols].to_string(index=False))

In [ ]:
# Tax Increase Summary
print("\n" + "="*80)
print("TAX RETURNS WITH AN INCREASE IN LIABILITY")
print("="*80)
increase_cols = ["Federal AGI Range", "Increase #", "Increase %", "Total Increase ($)", "Avg Increase"]
print(df_results[increase_cols].to_string(index=False))

In [ ]:
# No Change and Zero Tax
print("\n" + "="*80)
print("TAX RETURNS WITH NO CHANGE / ZERO TAX LIABILITY")
print("="*80)
other_cols = ["Federal AGI Range", "No Change #", "No Change %", "Zero Tax #", "Zero Tax %"]
print(df_results[other_cols].to_string(index=False))

## Comparison to RFA Fiscal Note

The SC Revenue & Fiscal Affairs (RFA) Office estimated H.4216 would have a **-$119.1M** General Fund impact.

Key differences between PolicyEngine and RFA estimates:
- **Population**: PE counts all tax units (filers + non-filers); RFA counts only actual filers
- **Data source**: PE uses CPS-based synthetic data; RFA uses actual SC tax return data
- **Income distribution**: PE has different return counts by income bracket, particularly more millionaires

In [ ]:
# Load RFA analysis for comparison
rfa_df = pd.read_csv('rfa_h4216_analysis.csv')

print("="*80)
print("COMPARISON: PolicyEngine vs RFA Fiscal Note")
print("="*80)

# RFA total impact
rfa_total_impact = rfa_df['Total Change'].sum()
pe_total_impact = total_change_amount

print(f"\nGeneral Fund Impact:")
print(f"  RFA Estimate:         ${rfa_total_impact:>15,.0f}")
print(f"  PolicyEngine Estimate: ${pe_total_impact:>15,.0f}")
print(f"  Difference:           ${pe_total_impact - rfa_total_impact:>15,.0f}")

# Calculate accuracy
accuracy = 1 - abs(pe_total_impact - rfa_total_impact) / abs(rfa_total_impact)
print(f"\n  Accuracy vs RFA: {accuracy*100:.1f}%")

# Return count comparison
rfa_total_returns = rfa_df['Est. # of Returns'].sum()
print(f"\nTotal Returns:")
print(f"  RFA:          {rfa_total_returns:>12,.0f}")
print(f"  PolicyEngine: {int(total_returns):>12,.0f}")
print(f"  Difference:   {int(total_returns - rfa_total_returns):>+12,.0f}")

In [ ]:
# Side-by-side comparison by income bracket
print("\n" + "="*80)
print("IMPACT BY INCOME BRACKET: PolicyEngine vs RFA")
print("="*80)

# Map PE brackets to RFA brackets for comparison
bracket_comparison = []
for idx, row in df_results.iterrows():
    if row['Federal AGI Range'] == 'Total':
        continue
    
    # Find matching RFA row
    rfa_match = rfa_df[rfa_df['Federal AGI Range'] == row['Federal AGI Range']]
    if len(rfa_match) > 0:
        rfa_impact = rfa_match['Total Change'].values[0]
        rfa_returns = rfa_match['Est. # of Returns'].values[0]
    else:
        rfa_impact = 0
        rfa_returns = 0
    
    bracket_comparison.append({
        'AGI Range': row['Federal AGI Range'],
        'PE Returns': row['Est. # Returns'],
        'RFA Returns': rfa_returns,
        'PE Impact': row['Total Change ($)'],
        'RFA Impact': rfa_impact,
        'Diff ($)': row['Total Change ($)'] - rfa_impact
    })

comparison_df = pd.DataFrame(bracket_comparison)
print(comparison_df.to_string(index=False))